In [12]:
# !pip install langgraph-supervisor

In [13]:
import getpass
import os


if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google AI API key: ")

In [15]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.7,  # Gemini 3.0+ defaults to 1.0
    google_api_key=os.getenv("GOOGLE_API_KEY"),
    # other params...
)

In [16]:
from langgraph.prebuilt import create_react_agent
from langgraph_supervisor import create_supervisor

from langchain_openai import ChatOpenAI

from langgraph.func import entrypoint, task
from langgraph.graph import add_messages

# model = ChatOpenAI(model="gpt-4o")
model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.7,  # Gemini 3.0+ defaults to 1.0
    google_api_key=os.getenv("GOOGLE_API_KEY"),
    # other params...
)

# Create specialized agents

# Functional API - Agent 1 (Joke Generator)
@task
def generate_joke(messages):
    """First LLM call to generate initial joke"""
    system_message = {
        "role": "system", 
        "content": "Write a short joke"
    }
    msg = model.invoke(
        [system_message] + messages
    )
    return msg

@entrypoint()
def joke_agent(state):
    joke = generate_joke(state['messages']).result()
    messages = add_messages(state["messages"], [joke])
    return {"messages": messages}

joke_agent.name = "joke_agent"

# Graph API - Agent 2 (Research Expert)
def web_search(query: str) -> str:
    """Search the web for information."""
    return (
        "Here are the headcounts for each of the FAANG companies in 2024:\n"
        "1. **Facebook (Meta)**: 67,317 employees.\n"
        "2. **Apple**: 164,000 employees.\n"
        "3. **Amazon**: 1,551,000 employees.\n"
        "4. **Netflix**: 14,000 employees.\n"
        "5. **Google (Alphabet)**: 181,269 employees."
    )

research_agent = create_react_agent(
    model=model,
    tools=[web_search],
    name="research_expert",
    prompt="You are a world class researcher with access to web search. Do not do any math."
)

# Create supervisor workflow
workflow = create_supervisor(
    [research_agent, joke_agent],
    model=model,
    prompt=(
        "You are a team supervisor managing a research expert and a joke expert. "
        "For current events, use research_agent. "
        "For any jokes, use joke_agent."
    )
)

# Compile and run
app = workflow.compile()
result = app.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Share a joke to relax and start vibe coding for my next project idea."
        }
    ]
})

for m in result["messages"]:
    m.pretty_print()

/var/folders/vz/zs883bb125v6xmzdr27py7p40000gn/T/ipykernel_2047/3023901800.py:52: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  research_agent = create_react_agent(


================================ Human Message =================================

Share a joke to relax and start vibe coding for my next project idea.
================================== Ai Message ==================================
Name: supervisor
Tool Calls:
  transfer_to_joke_agent (56eeb640-6ff9-494b-8cbd-ad7bb13f2871)
 Call ID: 56eeb640-6ff9-494b-8cbd-ad7bb13f2871
  Args:
================================= Tool Message =================================
Name: transfer_to_joke_agent

Successfully transferred to joke_agent
================================== Ai Message ==================================

Okay, here's one to get you warmed up:

Why was the JavaScript developer sad?

Because he didn't Node how to Express himself.

Hope that gets the creative juices flowing for your next project! Happy coding!
================================== Ai Message ==================================
Name: joke_agent

Transferring back to supervisor
Tool Calls:
  transfer_back_to_supervisor (b5d22a